# 04. 그래프 데이터베이스(Graph DB) 입문 — Kùzu 로 소재 지식그래프 만들기

앞에서 관계형 DB(RDB, 표)와 벡터 DB(의미 검색)를 배웠습니다. 이번에는 **"무엇이 무엇과 어떻게 연결되어 있는가"** 를 그대로 저장하는 **그래프 데이터베이스(Graph DB)** 를 배웁니다.
이 노트북에서는 **LLM 을 전혀 쓰지 않습니다.** 손으로 작은 소재 지식그래프를 만들고, 그래프 질의 언어 **Cypher** 로 질문해 봅니다. (LLM 이 그래프를 자동으로 만들어 주는 과정은 `07_Graph_RAG` 에서 다룹니다.)

**학습 목표**
1. 표(RDB)와 그래프의 차이, 그래프가 유리한 질문 유형을 설명할 수 있습니다.
2. 노드(Node)·관계(Relationship)·속성(Property)으로 이루어진 **속성 그래프(Labeled Property Graph)** 를 이해합니다.
3. 논문 내용을 바탕으로 작은 **소재 지식그래프** 를 직접 설계하고 Kùzu 에 저장합니다.
4. Cypher 의 `CREATE / MERGE / MATCH / WHERE / RETURN`, 경로 패턴, 다단계 탐색(hop), 집계를 사용할 수 있습니다.
5. 같은 질문을 SQL(JOIN)과 Cypher 로 각각 써 보고 차이를 체감합니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import shutil
import kuzu                      # 그래프 데이터베이스 (파이썬 안에서 바로 돌아가는 '임베디드' 방식)
import pandas as pd
from skku_rag import DB_DIR      # 데이터베이스 파일을 모아 두는 폴더 (repo/db)

print("kuzu 버전:", kuzu.__version__)

## 1. 왜 그래프인가?

소재 연구의 지식은 대부분 **연결** 로 이루어져 있습니다.

```
   [재료] ──가진다──▶ [물성] ◀──증가시킨다── [현상]
     │                                        ▲
   대체 후보                                 원인
     ▼                                        │
   [재료] ──포함한다──▶ [원소]          [다른 현상]
```

- **상평형도** 를 떠올려 보세요. 하나의 점(조성·온도)이 어느 상(phase)과 이웃하고, 어느 경계를 넘으면 어떤 상으로 바뀌는지 — 이 "이웃 관계" 자체가 정보입니다.
- **논문 인용망** 도 마찬가지입니다. 논문 A 가 B 를 인용하고, B 가 C 를 인용한다면 "A 와 C 는 2단계로 연결" 되어 있습니다.

이런 질문을 생각해 봅시다.

> **"Cu 를 대체할 후보 중, Al 을 포함하는 화합물은 무엇이고, 그 화합물의 다른 구성 원소의 전자 평균자유행로(EMFP)는 얼마인가?"**

표(RDB)로 저장했다면 `재료표 ⋈ 대체관계표 ⋈ 구성원소표 ⋈ 물성표 ...` 처럼 **JOIN 을 여러 번** 해야 합니다. 그래프에서는 "Cu ← 대체후보 → Al 포함 → 다른 원소 → EMFP" 라는 **길(경로)을 따라가기만** 하면 됩니다.

| 비교 | 관계형 DB (RDB) | 그래프 DB |
|---|---|---|
| 저장 단위 | 행(row)과 열(column)로 된 **표** | **노드**(점)와 **관계**(선) |
| 연결 표현 | 외래키 + 연결용 표, 질의할 때 JOIN | 관계가 **직접 저장** 되어 있음 |
| 잘하는 질문 | 집계·필터 ("비저항 < 5 인 금속 목록") | 연결·경로 ("A 와 2단계 안에 연결된 것", "A 에서 B 로 가는 길") |
| 비유 | 물성표(엑셀) | 상평형도·인용 관계도 |

## 2. 속성 그래프(Labeled Property Graph)의 세 가지 구성 요소

| 요소 | 뜻 | 소재 예시 |
|---|---|---|
| **노드(Node)** + 레이블(Label) | 대상(점). 레이블은 노드의 '종류' | `(:Material {name: "Cu"})`, `(:Phenomenon {name: "size effect"})` |
| **관계(Relationship)** + 타입(Type) | 두 노드를 잇는 **방향이 있는** 선 | `(Ru)-[:ALTERNATIVE_TO]->(Cu)` "Ru 는 Cu 의 대체 후보" |
| **속성(Property)** | 노드나 관계에 붙는 `이름: 값` 정보 | 노드 `{crystal: "fcc"}`, 관계 `{value: 39.9}` |

Cypher 에서는 그림 그리듯이 씁니다. **괄호 `( )` 는 노드, 화살표 `-[ ]->` 는 관계** 입니다.

```
(ru:Material {name: "Ru"}) -[:ALTERNATIVE_TO {reason: "짧은 EMFP"}]-> (cu:Material {name: "Cu"})
 └─ 노드(변수:레이블 {속성}) ─┘ └────────── 관계(타입 {속성}) ──────────┘ └──────── 노드 ────────┘
```

> 참고: 노드 이름은 영어로 적습니다. 논문 원문이 영어이고, 나중에 그래프를 그릴 때 한글 글꼴이 없는 환경(Colab)에서도 글자가 깨지지 않기 때문입니다.

## 3. 소재 지식그래프 설계하기 (논문 내용 기반)

`paper_data/` 의 두 논문에서 몇 가지 사실을 골라 그래프로 옮겨 보겠습니다. (수치는 리뷰 논문 Sung et al. 2026 의 표, Kim et al. 2016 본문에서 가져왔습니다.)

- Cu 는 벌크 비저항이 가장 낮지만(1.678 μΩ·cm) **EMFP 가 약 39.9 nm 로 길어서**, 배선이 수십 nm 이하로 얇아지면 **크기효과(size effect)** 로 비저항이 급격히 커집니다.
- 크기효과의 원인은 **결정립계 산란(grain boundary scattering)** 과 **표면 산란(surface scattering)** 입니다.
- 대체 후보: **Ru, Co, Mo**(짧은 EMFP), 금속간화합물 **NiAl, RuAl, CuAl2**(얇아져도 비저항 증가가 완만).
- Kim 2016: 묽은 나노결정 **Cu-Nb, Cu-Fe** 합금에서 용질의 **결정립계 편석(GB segregation)** 이 **결정립 조대화(grain coarsening)** 와 비저항에 영향을 줍니다.

**설계도 (스키마)**

```
 노드 테이블                            관계 테이블
 ─────────────────────────────          ──────────────────────────────────────────────
 Material  (name, category, crystal)    HAS_PROPERTY   Material  → Property  {value}
 Property  (name, unit)                 ALTERNATIVE_TO Material  → Material  {reason}
 Phenomenon(name)                       CONTAINS       Material  → Material
 Paper     (id, title, year)            SHOWS          Material  → Phenomenon
                                        CAUSES         Phenomenon→ Phenomenon
                                        INCREASES      Phenomenon→ Property
                                        REPORTED_IN    Material  → Paper
```

## 4. Kùzu 데이터베이스 만들기

**Kùzu** 는 `pip install kuzu` 만으로 쓸 수 있는 **임베디드(embedded) 그래프 DB** 입니다. 별도 서버 없이 **파일 하나** 에 그래프를 저장합니다 (SQLite 의 그래프 버전이라고 생각하면 됩니다).

> 참고: Kùzu 는 2025년 10월 개발이 종료되어 저장소가 보관(archive) 상태입니다. 기능이 '얼어 있는' 대신 **0.11.3 버전은 안정적으로 동작** 하므로 학습용으로는 충분합니다. 여기서 배우는 **Cypher 문법은 대표적인 그래프 DB 인 Neo4j 에서도 거의 그대로** 쓸 수 있습니다.

매번 깨끗한 상태에서 시작하도록, 기존 파일이 있으면 지우고 새로 만듭니다.

In [ ]:
db_path = DB_DIR / "toy_graph.kuzu"           # 그래프가 저장될 파일

# 셀을 다시 실행할 때: 열려 있던 연결·DB 를 먼저 닫아야 파일을 지울 수 있음 (Windows 파일 잠금)
if "conn" in globals():
    conn.close()
    db.close()

# 이전 실행 결과가 있으면 지웁니다 (항상 새로 시작)
for p in [db_path, DB_DIR / "toy_graph.kuzu.wal"]:
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

db = kuzu.Database(str(db_path))   # DB 파일 열기(없으면 생성)
conn = kuzu.Connection(db)         # 질의를 주고받는 연결 통로
print("DB 생성:", db_path.relative_to(DB_DIR.parent))

### 4-1. 스키마 정의: NODE TABLE / REL TABLE

Kùzu 는 Neo4j 와 달리 **먼저 스키마(설계도)를 정해야** 합니다. RDB 의 `CREATE TABLE` 과 비슷합니다.

- 왜 스키마가 필요할까요? 실험노트 양식을 미리 정해 두면 "단위를 빼먹은 기록"이 생기지 않듯, **어떤 노드가 어떤 속성을 갖는지, 어떤 관계가 어느 노드끼리 이어지는지** 를 DB 가 검사해 줍니다. 또 구조를 알고 있으니 저장·검색이 빨라집니다.
- RDB 와의 차이: 관계(REL TABLE)가 **"FROM 어떤 노드 TO 어떤 노드"** 로 정의되어, 연결 자체가 1급 시민으로 저장됩니다. (RDB 에서는 외래키 열을 만들고 JOIN 으로 이어 붙여야 합니다.)

In [ ]:
schema = [
    # 노드 테이블: 이름(name)이 기본키(PRIMARY KEY) → 같은 이름의 노드는 하나만 존재
    "CREATE NODE TABLE Material(name STRING, category STRING, crystal STRING, PRIMARY KEY(name))",
    "CREATE NODE TABLE Property(name STRING, unit STRING, PRIMARY KEY(name))",
    "CREATE NODE TABLE Phenomenon(name STRING, PRIMARY KEY(name))",
    "CREATE NODE TABLE Paper(id STRING, title STRING, year INT64, PRIMARY KEY(id))",
    # 관계 테이블: FROM (출발 노드 종류) TO (도착 노드 종류), 관계에도 속성을 붙일 수 있음
    "CREATE REL TABLE HAS_PROPERTY(FROM Material TO Property, value DOUBLE)",
    "CREATE REL TABLE ALTERNATIVE_TO(FROM Material TO Material, reason STRING)",
    "CREATE REL TABLE CONTAINS(FROM Material TO Material)",
    "CREATE REL TABLE SHOWS(FROM Material TO Phenomenon)",
    "CREATE REL TABLE CAUSES(FROM Phenomenon TO Phenomenon)",
    "CREATE REL TABLE INCREASES(FROM Phenomenon TO Property)",
    "CREATE REL TABLE REPORTED_IN(FROM Material TO Paper)",
]
for stmt in schema:
    conn.execute(stmt)
conn.execute("CALL show_tables() RETURN name, type").get_as_df()

## 5. Cypher 기초 ① — 노드 만들기: `CREATE` 와 `MERGE`

- `CREATE` : 무조건 새로 만듭니다.
- `MERGE` : **있으면 그대로, 없으면 만듭니다.** (같은 재료를 두 번 넣어도 중복이 생기지 않음 → 실무에서 주로 사용)
- `$name` 처럼 `$` 가 붙은 부분은 **매개변수(parameter)** 입니다. 값을 문자열에 직접 끼워 넣지 않고 따로 전달하면 따옴표 실수나 보안 문제를 피할 수 있습니다.

In [ ]:
# (이름, 분류, 결정구조)
materials = [
    ("Cu", "metal", "fcc"), ("Ru", "metal", "hcp"), ("Co", "metal", "hcp"),
    ("Mo", "metal", "bcc"), ("Al", "metal", "fcc"), ("Ni", "metal", "fcc"),
    ("Nb", "metal", "bcc"), ("Fe", "metal", "bcc"),
    ("NiAl", "intermetallic", "B2"), ("RuAl", "intermetallic", "B2"), ("CuAl2", "intermetallic", "tetragonal"),
    ("Cu-Nb", "dilute alloy", "fcc"), ("Cu-Fe", "dilute alloy", "fcc"),
]
for name, cat, crystal in materials:
    conn.execute("CREATE (:Material {name: $name, category: $cat, crystal: $crystal})",
                 {"name": name, "cat": cat, "crystal": crystal})
print(conn.execute("MATCH (m:Material) RETURN count(m) AS 재료수").get_as_df())

In [ ]:
# MERGE: 이미 있는 'Cu' 를 다시 넣어도 개수가 늘지 않습니다
conn.execute("MERGE (m:Material {name: 'Cu'})")
print("MERGE 후 재료 수:", conn.execute("MATCH (m:Material) RETURN count(m)").get_as_df().iloc[0, 0])

# 참고: CREATE 로 같은 이름을 또 넣으면 기본키 중복 오류가 납니다
try:
    conn.execute("CREATE (:Material {name: 'Cu'})")
except RuntimeError as e:
    print("CREATE 오류:", str(e)[:80])

In [ ]:
# 물성, 현상, 논문 노드
for name, unit in [("resistivity", "uOhm.cm"), ("EMFP", "nm"), ("cohesive energy", "eV/atom")]:
    conn.execute("MERGE (p:Property {name: $n}) SET p.unit = $u", {"n": name, "u": unit})
for name in ["size effect", "grain boundary scattering", "surface scattering",
             "electromigration", "GB segregation", "grain coarsening"]:
    conn.execute("MERGE (:Phenomenon {name: $n})", {"n": name})
papers = [("kim2016", "Interplay between GB segregation and resistivity in dilute nanocrystalline Cu alloys", 2016),
          ("sung2026", "Recent progress in alternative metals for advanced interconnects", 2026)]
for pid, title, year in papers:
    conn.execute("MERGE (p:Paper {id: $id}) SET p.title = $t, p.year = $y", {"id": pid, "t": title, "y": year})

## 6. Cypher 기초 ② — 관계 만들기: `MATCH ... CREATE`

관계를 만들려면 먼저 양쪽 노드를 `MATCH` 로 **찾은 다음**, 그 둘 사이에 화살표를 그립니다.

```
MATCH (a:Material {name: 'Ru'}), (b:Material {name: 'Cu'})   ← 두 노드를 찾고
CREATE (a)-[:ALTERNATIVE_TO {reason: '...'}]->(b)             ← 화살표로 잇기
```

관계 종류마다 같은 모양의 쿼리를 반복하므로, 작은 함수 `link()` 를 만들어 씁니다.

In [ ]:
def link(src_label, src, rel, dst_label, dst, props=None):
    # 관계 속성(props)이 있으면 {key: $key, ...} 형태로 붙입니다
    props = props or {}
    prop_txt = ", ".join(f"{k}: ${k}" for k in props)  # 예) "value: $value"
    query = (f"MATCH (a:{src_label} {{name: $src}}), (b:{dst_label} {{name: $dst}}) "
             f"CREATE (a)-[:{rel} {{{prop_txt}}}]->(b)")
    conn.execute(query, {"src": src, "dst": dst, **props})

In [ ]:
# ① 물성 값 (Sung 2026 표: 벌크 비저항 μΩ·cm, EMFP nm / 본문: 응집에너지 eV/atom)
props = {"Cu": (1.678, 39.9, 3.5), "Co": (6.2, 11.8, 5.2), "Ru": (7.8, 6.59, None),
         "Mo": (5.34, 11.2, 6.8), "Al": (2.65, 18.9, None), "Ni": (6.93, 5.87, None)}
for m, (rho, emfp, ecoh) in props.items():
    link("Material", m, "HAS_PROPERTY", "Property", "resistivity", {"value": rho})
    link("Material", m, "HAS_PROPERTY", "Property", "EMFP", {"value": emfp})
    if ecoh is not None:
        link("Material", m, "HAS_PROPERTY", "Property", "cohesive energy", {"value": ecoh})

In [ ]:
# ② Cu 대체 후보와 그 이유
alternatives = {"Ru": "short EMFP, strong EM resistance", "Co": "short EMFP",
                "Mo": "high cohesive energy", "NiAl": "gentle resistivity scaling",
                "RuAl": "barrier-free candidate", "CuAl2": "outperforms Cu below 10 nm"}
for m, reason in alternatives.items():
    link("Material", m, "ALTERNATIVE_TO", "Material", "Cu", {"reason": reason})

# ③ 화합물·합금의 구성 원소
for compound, elements in {"NiAl": ["Ni", "Al"], "RuAl": ["Ru", "Al"], "CuAl2": ["Cu", "Al"],
                           "Cu-Nb": ["Cu", "Nb"], "Cu-Fe": ["Cu", "Fe"]}.items():
    for el in elements:
        link("Material", compound, "CONTAINS", "Material", el)

In [ ]:
# ④ 현상 사이의 인과, 현상 → 물성
link("Phenomenon", "grain boundary scattering", "CAUSES", "Phenomenon", "size effect")
link("Phenomenon", "surface scattering", "CAUSES", "Phenomenon", "size effect")
link("Phenomenon", "GB segregation", "CAUSES", "Phenomenon", "grain coarsening")
link("Phenomenon", "size effect", "INCREASES", "Property", "resistivity")
link("Phenomenon", "grain boundary scattering", "INCREASES", "Property", "resistivity")

# ⑤ 재료가 보이는 현상, 재료를 다룬 논문
for m, ph in [("Cu", "size effect"), ("Cu", "electromigration"),
              ("Cu-Nb", "GB segregation"), ("Cu-Fe", "GB segregation")]:
    link("Material", m, "SHOWS", "Phenomenon", ph)
for m in ["Cu", "Cu-Nb", "Cu-Fe"]:
    conn.execute("MATCH (a:Material {name:$m}), (p:Paper {id:'kim2016'}) CREATE (a)-[:REPORTED_IN]->(p)", {"m": m})
for m in ["Cu", "Ru", "Co", "Mo", "NiAl", "RuAl", "CuAl2"]:
    conn.execute("MATCH (a:Material {name:$m}), (p:Paper {id:'sung2026'}) CREATE (a)-[:REPORTED_IN]->(p)", {"m": m})

## 7. Cypher 기초 ③ — 질문하기: `MATCH / WHERE / RETURN`

Cypher 질의는 **"이런 모양의 패턴을 찾아(MATCH), 조건으로 거르고(WHERE), 원하는 것을 돌려줘(RETURN)"** 입니다.
결과는 `.get_as_df()` 로 pandas 표(DataFrame)로 받아 보기 좋게 출력합니다.

In [ ]:
# 모든 재료의 이름과 결정구조
conn.execute('''
    MATCH (m:Material)
    RETURN m.name AS 재료, m.category AS 분류, m.crystal AS 결정구조
    ORDER BY 분류, 재료
''').get_as_df()

In [ ]:
# WHERE: EMFP 가 15 nm 보다 짧은 재료 (관계 속성 r.value 로 거르기)
conn.execute('''
    MATCH (m:Material)-[r:HAS_PROPERTY]->(p:Property {name: 'EMFP'})
    WHERE r.value < 15
    RETURN m.name AS 재료, r.value AS EMFP_nm
    ORDER BY EMFP_nm
''').get_as_df()

### 7-1. 경로 패턴 `(a)-[:R]->(b)`

화살표 방향이 곧 의미입니다. `(x)-[:ALTERNATIVE_TO]->(cu)` 는 "x 가 cu 의 대체 후보" 라는 뜻입니다.

In [ ]:
# Cu 를 대체할 후보와 그 이유
conn.execute('''
    MATCH (x:Material)-[r:ALTERNATIVE_TO]->(cu:Material {name: 'Cu'})
    RETURN x.name AS 후보, x.category AS 분류, r.reason AS 이유
''').get_as_df()

In [ ]:
# 패턴을 이어 붙이기: Cu 대체 후보 중 Al 을 포함하는 화합물
conn.execute('''
    MATCH (cu:Material {name: 'Cu'})<-[:ALTERNATIVE_TO]-(x:Material)-[:CONTAINS]->(al:Material {name: 'Al'})
    RETURN x.name AS Al_포함_후보
''').get_as_df()

### 7-2. 다단계 탐색(Multi-hop) `[*1..2]`

`-[*1..2]-` 는 "**관계를 1번 또는 2번** 건너서" 라는 뜻입니다. 관계 타입을 지정하지 않으면 모든 종류의 관계를 따라갑니다. (화살표 `>` 를 빼면 방향도 무시)
논문 인용망에서 "내 논문과 2단계 안에 연결된 논문"을 찾는 것과 같은 질문입니다.

In [ ]:
# Cu 에서 2단계 안에 연결된 모든 노드 (종류별로)
# - label(n): 노드 종류, coalesce(a, b): a 가 비어 있으면 b (Paper 노드는 name 대신 id 를 가짐)
# - WITH: 중간 결과에 이름을 붙여 다음 단계로 넘김 / Cu→Ru→Cu 처럼 제자리로 돌아오는 경로는 WHERE 로 제외
conn.execute('''
    MATCH (cu:Material {name: 'Cu'})-[e*1..2]-(n)
    WITH label(n) AS kind, coalesce(n.name, n.id) AS name, length(e) AS hop
    WHERE name <> 'Cu'
    RETURN kind AS 종류, name AS 이름, min(hop) AS 최소_hop
    ORDER BY 최소_hop, 종류, 이름
''').get_as_df()

In [ ]:
# 인과 사슬: 비저항을 증가시키는 현상과, 그 현상을 일으키는 원인 현상까지
conn.execute('''
    MATCH (cause:Phenomenon)-[:CAUSES*0..1]->(ph:Phenomenon)-[:INCREASES]->(:Property {name: 'resistivity'})
    RETURN DISTINCT cause.name AS 원인, ph.name AS 비저항을_높이는_현상
''').get_as_df()

> `*0..1` 은 "0번 또는 1번" 이라는 뜻입니다. 0번이면 `cause` 와 `ph` 가 같은 노드가 되므로, 비저항을 **직접** 높이는 현상도 함께 나옵니다.

### 7-3. 집계(Aggregation)

SQL 의 `GROUP BY` 처럼 `count()`, `avg()`, `collect()`(목록으로 모으기)를 쓸 수 있습니다. Cypher 는 `GROUP BY` 를 따로 쓰지 않고, **RETURN 에 집계 함수가 아닌 열이 그룹 기준** 이 됩니다.

In [ ]:
# 논문별로 다룬 재료 수와 목록
conn.execute('''
    MATCH (m:Material)-[:REPORTED_IN]->(p:Paper)
    RETURN p.id AS 논문, p.year AS 연도, count(m) AS 재료수, collect(m.name) AS 재료목록
    ORDER BY 연도
''').get_as_df()

In [ ]:
# Cu 와 대체 후보 금속의 EMFP 비교 (평균도 함께)
conn.execute('''
    MATCH (x:Material)-[:ALTERNATIVE_TO]->(:Material {name: 'Cu'}),
          (x)-[r:HAS_PROPERTY]->(:Property {name: 'EMFP'})
    RETURN count(x) AS 후보수, avg(r.value) AS 후보_평균_EMFP_nm, collect(x.name) AS 후보
''').get_as_df()

## 8. 그래프 그려 보기 (networkx + matplotlib)

DB 에 저장된 **모든 관계** 를 꺼내 networkx 그래프로 옮긴 뒤 그립니다. `label(r)` 은 관계 타입, `label(n)` 은 노드 종류를 알려 줍니다.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

edges = conn.execute('''
    MATCH (a)-[r]->(b)
    RETURN coalesce(a.name, a.id) AS src, label(a) AS src_kind, label(r) AS rel,
           coalesce(b.name, b.id) AS dst, label(b) AS dst_kind
''').get_as_df()
print("관계 수:", len(edges))
edges.head()

In [ ]:
G = nx.DiGraph()
for row in edges[edges.rel != "REPORTED_IN"].itertuples():   # 그림이 복잡해지지 않게 논문 연결은 제외
    G.add_node(row.src, kind=row.src_kind)
    G.add_node(row.dst, kind=row.dst_kind)
    G.add_edge(row.src, row.dst, rel=row.rel)

colors = {"Material": "#4C9BE8", "Property": "#F2A541", "Phenomenon": "#E5625E"}
node_colors = [colors[G.nodes[n]["kind"]] for n in G.nodes]
pos = nx.spring_layout(G, seed=3, k=1.6)          # seed 고정 → 매번 같은 배치

plt.figure(figsize=(13, 9))
nx.draw_networkx(G, pos, node_color=node_colors, node_size=900, font_size=8, arrowsize=12, edge_color="#999")
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "rel"), font_size=6)
for kind, c in colors.items():                     # 범례
    plt.scatter([], [], color=c, label=kind)
plt.legend(loc="lower left"); plt.axis("off"); plt.title("Toy materials knowledge graph"); plt.show()

## 9. 같은 질문, SQL 과 Cypher 비교

질문: **"Cu 의 대체 후보 화합물 중 Al 을 포함하는 것에 대해, 그 화합물의 *다른* 구성 원소와 그 원소의 EMFP 를 보여줘."**

같은 데이터를 RDB 표로 옮기면 다음 4개 표가 필요합니다.

```
material(name, category)            alternative(candidate, target, reason)
contains(compound, element)         property(material, prop, value)
```

SQLite(파이썬 내장)로 실제로 만들어서 두 방식을 나란히 실행해 보겠습니다.

In [ ]:
import sqlite3

sql = sqlite3.connect(":memory:")     # 메모리 안에 임시 RDB 만들기
_ = sql.executescript('''
    CREATE TABLE alternative(candidate TEXT, target TEXT);
    CREATE TABLE contains(compound TEXT, element TEXT);
    CREATE TABLE property(material TEXT, prop TEXT, value REAL);
''')
# 그래프에 넣었던 것과 같은 데이터를 표에 복사
sql.executemany("INSERT INTO alternative VALUES (?, 'Cu')", [(m,) for m in alternatives])
sql.executemany("INSERT INTO contains VALUES (?, ?)",
                [(c, e) for c, els in {"NiAl": ["Ni", "Al"], "RuAl": ["Ru", "Al"], "CuAl2": ["Cu", "Al"]}.items() for e in els])
sql.executemany("INSERT INTO property VALUES (?, 'EMFP', ?)", [(m, v[1]) for m, v in props.items()])
sql.commit()

In [ ]:
# SQL: JOIN 을 4번 — 같은 contains 표를 두 번(c1: Al 포함 확인, c2: 다른 원소) 붙여야 합니다
sql_query = '''
SELECT a.candidate AS compound, c2.element AS other_element, p.value AS EMFP_nm
FROM alternative a
JOIN contains c1 ON c1.compound = a.candidate AND c1.element = 'Al'
JOIN contains c2 ON c2.compound = a.candidate AND c2.element <> 'Al'
LEFT JOIN property p ON p.material = c2.element AND p.prop = 'EMFP'
WHERE a.target = 'Cu'
ORDER BY compound
'''
pd.read_sql(sql_query, sql)

In [ ]:
# Cypher: 질문 문장을 그대로 '그림'으로 옮긴 모양
cypher_query = '''
MATCH (:Material {name:'Cu'})<-[:ALTERNATIVE_TO]-(x:Material)-[:CONTAINS]->(:Material {name:'Al'}),
      (x)-[:CONTAINS]->(other:Material)
WHERE other.name <> 'Al'
OPTIONAL MATCH (other)-[r:HAS_PROPERTY]->(:Property {name:'EMFP'})
RETURN x.name AS compound, other.name AS other_element, r.value AS EMFP_nm
ORDER BY compound
'''
conn.execute(cypher_query).get_as_df()

두 결과가 같은지 확인해 보세요. 결과가 같다면, 차이는 **읽기 쉬움** 과 **확장성** 입니다.

| | SQL | Cypher |
|---|---|---|
| 쿼리 모양 | 표 이름과 `ON` 조건을 머릿속에서 연결해야 함 | `(Cu)<-대체-(x)-포함->(Al)` 그림을 그대로 읽음 |
| 질문이 1단계 더 깊어지면 | JOIN 한 줄 + ON 조건 추가 | 화살표 하나 추가, 또는 `*1..3` 처럼 깊이만 바꿈 |
| "몇 단계인지 모르는" 질문 | 재귀 쿼리(WITH RECURSIVE)가 필요해 어려움 | `[*1..n]` 으로 간단 |

> 반대로 "비저항 평균", "연도별 개수" 같은 **표 형태 집계** 는 SQL 이 더 자연스럽습니다. 그래서 실제 시스템은 **RDB·벡터DB·그래프DB 를 함께** 쓰고, 질문에 맞는 도구를 고릅니다 (→ `08_Agentic_RAG`).

## 정리

- 그래프 DB 는 **노드(대상)·관계(연결)·속성(값)** 으로 지식을 저장합니다. 연결 자체가 저장되므로 "A 와 B 는 어떻게 이어져 있나?" 같은 질문에 강합니다.
- **Kùzu** 는 파일 하나로 동작하는 임베디드 그래프 DB 이며, `NODE TABLE` / `REL TABLE` 로 스키마를 먼저 정합니다.
- Cypher 핵심 문법

| 문법 | 의미 | 예 |
|---|---|---|
| `CREATE` / `MERGE` | 만들기 / 없을 때만 만들기 | `MERGE (:Material {name:'Cu'})` |
| `MATCH ... WHERE ... RETURN` | 패턴 찾기 → 거르기 → 돌려주기 | `MATCH (m:Material) WHERE ... RETURN m.name` |
| `(a)-[:R]->(b)` | 방향 있는 관계 패턴 | `(x)-[:ALTERNATIVE_TO]->(cu)` |
| `-[*1..2]-` | 1~2단계 건너뛰기 | 이웃의 이웃 찾기 |
| `count / avg / collect` | 집계 | 논문별 재료 수 |

- 다음 `07_Graph_RAG` 에서는 이 그래프를 **LLM 이 논문에서 자동으로 추출** 하게 하고, 질문에 답할 때 그래프를 검색에 활용합니다.

## 직접 해보기

1. `W`(텅스텐, bcc, 벌크 비저항 5.28 μΩ·cm, EMFP 15.5 nm)를 `MERGE` 로 추가하고, `ALTERNATIVE_TO` 관계로 Cu 와 연결해 보세요. 7-1 의 대체 후보 질의 결과가 어떻게 바뀌나요?
2. "**Cu-Nb 합금이 보이는 현상이 일으키는 현상**" 을 찾는 Cypher 를 작성해 보세요. (힌트: `(:Material {name:'Cu-Nb'})-[:SHOWS]->()-[:CAUSES]->(x)`)
3. 2단계(`*1..2`) 대신 3단계(`*1..3`)로 Cu 의 이웃을 찾으면 결과가 몇 개로 늘어나나요? 단계가 커질수록 결과가 폭발적으로 늘어나는 이유를 생각해 보세요.

## 참고자료
- Kùzu 문서 (Cypher 매뉴얼): https://docs.kuzudb.com/
- 공원나연, 「Neo4j GraphRAG 파이썬 패키지 가이드북」(WikiDocs): https://wikidocs.net/book/16760 — 그래프 개념과 Cypher 입문 흐름을 참고했습니다.
- 논문 데이터: Kim et al., *Scripta Materialia* 123 (2016) 113–117 / Sung et al., *J. Mater. Chem. C* 14 (2026) 6558–6580